# 1D-CNN Model Architecture and Training

Model architecture discussion

The model definition is in `form_analyzer.py`

I chose a CNN to do form analysis because the keypoint data is a timeseries. The motion of each point in time can represent each error. 

In [6]:
import torch   
device = torch.device("mps")

In [7]:
from form_analyzer import create_dataloaders
train_loader, val_loader, test_loader = create_dataloaders("datasets/custom_running_form/custom_labels.csv")

Dataset Split -> Train: 553 | Val: 118 | Test: 119


In [8]:
keypoint_path = "datasets/custom_running_form/keypoints"

def train_model(model, optimizer, loss_fn, train_loader, val_loader, num_epochs=20, patience=0, model_save_path="runs/form/weights/form_analyzer_model.pt"):
    best_val_loss = float('inf')

    patience_count = 0
    prev_val_loss = 1
    
    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0.0
        
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)

            optimizer.zero_grad()
            
            outputs = model(features)
            
            loss = loss_fn(outputs, labels)

            loss.backward()
            
            optimizer.step()
            
            running_train_loss += loss.item() * features.size(0)
            
        epoch_train_loss = running_train_loss / len(train_loader.dataset)
        

        model.eval()
        running_val_loss = 0.0
        all_preds = []
        all_labels = []
        
        # Disable gradient tracking for evaluation to save memory and speed up computation
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                
                outputs = model(features)
                loss = loss_fn(outputs, labels)
                running_val_loss += loss.item() * features.size(0)
                
                probs = torch.sigmoid(outputs)
                
                preds = (probs > 0.5).float()
                
                all_preds.append(preds.cpu())
                all_labels.append(labels.cpu())
                
        epoch_val_loss = running_val_loss / len(val_loader.dataset)

        if epoch_val_loss > prev_val_loss:
            patience_count += 1
            if patience_count == patience:
                print("Patience exhausted!")
                break
        else:
            patience_count = 0
        
        # Calculate Exact Match Accuracy 
        # (For a sample to be "correct", all 4 labels must match exactly)
        all_preds = torch.cat(all_preds)
        all_labels = torch.cat(all_labels)
        exact_match_acc = (all_preds == all_labels).all(dim=1).float().mean().item()
        
        print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Exact Match Acc: {exact_match_acc * 100:.2f}%")
        
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            print("Lowest val loss achieved, saving model.")
            torch.save(model.state_dict(), model_save_path)
            
    print("Training complete.")

In [9]:
import torch.nn as nn
import torch.optim as optim
from form_analyzer import FormAnalyzer1DCNN

learning_rate = 0.001

model_5kernel = FormAnalyzer1DCNN(kernel_size=5).to(device)

optimizer = optim.Adam(model_5kernel.parameters(), lr=learning_rate)
loss_fn = nn.BCEWithLogitsLoss()

model_save_path = "runs/form/weights/form_analyzer_model_5kernel.pt"

train_model(model_5kernel, optimizer, loss_fn, train_loader, val_loader, num_epochs=100, patience=3, model_save_path=model_save_path)

Epoch [01/100] | Train Loss: 0.5773 | Val Loss: 0.5487 | Val Exact Match Acc: 24.58%
Lowest val loss achieved, saving model.
Epoch [02/100] | Train Loss: 0.4336 | Val Loss: 0.3777 | Val Exact Match Acc: 41.53%
Lowest val loss achieved, saving model.
Epoch [03/100] | Train Loss: 0.3005 | Val Loss: 0.2297 | Val Exact Match Acc: 70.34%
Lowest val loss achieved, saving model.
Epoch [04/100] | Train Loss: 0.2508 | Val Loss: 0.3487 | Val Exact Match Acc: 44.07%
Epoch [05/100] | Train Loss: 0.1974 | Val Loss: 0.3077 | Val Exact Match Acc: 61.02%
Epoch [06/100] | Train Loss: 0.1827 | Val Loss: 0.1930 | Val Exact Match Acc: 57.63%
Lowest val loss achieved, saving model.
Epoch [07/100] | Train Loss: 0.1561 | Val Loss: 0.2262 | Val Exact Match Acc: 53.39%
Epoch [08/100] | Train Loss: 0.1316 | Val Loss: 0.3699 | Val Exact Match Acc: 55.08%
Epoch [09/100] | Train Loss: 0.1439 | Val Loss: 0.1667 | Val Exact Match Acc: 70.34%
Lowest val loss achieved, saving model.
Epoch [10/100] | Train Loss: 0.1113

In [10]:
learning_rate = 0.001

model_3kernel = FormAnalyzer1DCNN(kernel_size=3).to(device)

optimizer = optim.Adam(model_3kernel.parameters(), lr=learning_rate)
loss_fn = nn.BCEWithLogitsLoss()

model_save_path = "runs/form/weights/form_analyzer_model_3kernel.pt"

train_model(model_3kernel, optimizer, loss_fn, train_loader, val_loader, num_epochs=100, patience=3, model_save_path=model_save_path)

Epoch [01/100] | Train Loss: 0.5894 | Val Loss: 0.5704 | Val Exact Match Acc: 23.73%
Lowest val loss achieved, saving model.
Epoch [02/100] | Train Loss: 0.4274 | Val Loss: 0.3637 | Val Exact Match Acc: 35.59%
Lowest val loss achieved, saving model.
Epoch [03/100] | Train Loss: 0.2904 | Val Loss: 0.2628 | Val Exact Match Acc: 56.78%
Lowest val loss achieved, saving model.
Epoch [04/100] | Train Loss: 0.2184 | Val Loss: 0.2596 | Val Exact Match Acc: 61.86%
Lowest val loss achieved, saving model.
Epoch [05/100] | Train Loss: 0.1966 | Val Loss: 0.2729 | Val Exact Match Acc: 59.32%
Epoch [06/100] | Train Loss: 0.1907 | Val Loss: 0.2246 | Val Exact Match Acc: 59.32%
Lowest val loss achieved, saving model.
Epoch [07/100] | Train Loss: 0.1847 | Val Loss: 0.3154 | Val Exact Match Acc: 61.02%
Epoch [08/100] | Train Loss: 0.1555 | Val Loss: 0.1947 | Val Exact Match Acc: 66.95%
Lowest val loss achieved, saving model.
Epoch [09/100] | Train Loss: 0.1418 | Val Loss: 0.1597 | Val Exact Match Acc: 76